# 01 – Exploration : décès accidentels liés aux drogues (Connecticut, 2012-2022)

**Source** : Kaggle, « Accidental Drug Related Deaths 2012-2022 » (lien dans le README).
Chaque ligne correspond à un décès accidentel lié à une ou plusieurs substances détectées.

## Questions étudiées
1. Parmi les 5 substances les plus souvent détectées, quelle part des décès implique chacune, et comment cette part a-t-elle évolué de 2012 à 2022 ?
2. Quelles associations de substances reviennent le plus souvent parmi les décès ?
3. Quel est le profil (âge, sexe) des personnes décédées, et a-t-il changé entre 2012 et 2022 ?
4. Quels comtés comptent le plus de décès, en nombre et rapporté à leur population ?
5. Les décès surviennent-ils majoritairement au domicile ou à l'hôpital ?

## Ce que fait ce notebook
Charger le fichier brut et l'inspecter (taille, types, valeurs manquantes, catégories) **sans le modifier**.
Chaque constat sera noté ici, puis traité dans `02_nettoyage.ipynb`.

In [1]:
import pandas as pd

chemin_brut = "../data/raw/Accidental_Drug_Related_Deaths_2012-2022.csv"
deces_brut = pd.read_csv(chemin_brut)
deces_brut.shape

(10654, 48)

**Constat 1 - taille :** 10 654 décès et 48 colonnes. 

In [2]:
deces_brut.info()

<class 'pandas.DataFrame'>
RangeIndex: 10654 entries, 0 to 10653
Data columns (total 48 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Date                           10654 non-null  str    
 1   Date Type                      10654 non-null  str    
 2   Age                            10652 non-null  float64
 3   Sex                            10645 non-null  str    
 4   Race                           10625 non-null  str    
 5   Ethnicity                      1575 non-null   str    
 6   Residence City                 10148 non-null  str    
 7   Residence County               9486 non-null   str    
 8   Residence State                8756 non-null   str    
 9   Injury City                    10476 non-null  str    
 10  Injury County                  7332 non-null   str    
 11  Injury State                   7741 non-null   str    
 12  Injury Place                   10364 non-null  str    
 1

**Constat 2 :**

**Mauvais format:**
- Date est du texte 
- Age est en float

**Valeurs manquantes qui nous intéressent**
- Residence County n’a que 9 486 valeurs sur 10 654. Il manque donc environ 1 168 comtés, soit plus de 10 %
- Ethnicity n’est remplie que pour 1 575 lignes, donc presque vide.
- Location : 22 manquants
- Sex : 9 manquants. 
- Age : 2 manquants.

In [3]:
part_manquants = deces_brut.isna().mean().round(3)
part_manquants.sort_values(ascending=False)

Morphine (Not Heroin)            0.995
Hydromorphone                    0.994
Meth/Amphetamine                 0.988
Opiate NOS                       0.988
Other Opioid                     0.988
Oxymorphone                      0.985
Hydrocodone                      0.985
Tramad                           0.973
Gabapentin                       0.967
Amphet                           0.966
Other                            0.935
Heroin death certificate (DC)    0.931
Xylazine                         0.926
Fentanyl Analogue                0.925
Methadone                        0.912
Oxycodone                        0.911
Other Significant Conditions     0.905
Location if Other                0.888
Ethnicity                        0.852
Heroin/Morph/Codeine             0.804
Benzodiazepine                   0.765
Ethanol                          0.730
Heroin                           0.674
Cocaine                          0.638
Death State                      0.355
Fentanyl                 

**Constat 3 — Colonnes de substances : une case vide signifie « non détectée », pas « inconnue »**

Les colonnes de substances (de `Heroin` à `Other`) sont très peu remplies :
de 63,8 % de cases vides pour `Cocaine` à 99,5 % pour `Morphine (Not Heroin)`.
Ce n'est pas un défaut de collecte : ces colonnes signalent qu'une substance a été **détectée** (`Y`),
et restent vides sinon.

- **Hypothèse retenue** : case vide = substance non détectée. Au nettoyage, `Y` deviendra `True`
  et le vide `False`, pour compter les détections avec `sum`.
- **À vérifier d'abord** : que ces colonnes ne contiennent bien que `Y` et du vide.
- **Limite** : on ne distingue pas « recherchée et absente » de « non recherchée ».
  Les nombres de détections sont donc des minimums.

In [4]:
colonnes_substances = list(deces_brut.columns[22:44])
print(colonnes_substances)
for colonne in colonnes_substances:
    print(deces_brut[colonne].value_counts(dropna=False))
    print("-----")

['Heroin', 'Heroin death certificate (DC)', 'Cocaine', 'Fentanyl', 'Fentanyl Analogue', 'Oxycodone', 'Oxymorphone', 'Ethanol', 'Hydrocodone', 'Benzodiazepine', 'Methadone', 'Meth/Amphetamine', 'Amphet', 'Tramad', 'Hydromorphone', 'Morphine (Not Heroin)', 'Xylazine', 'Gabapentin', 'Opiate NOS', 'Heroin/Morph/Codeine', 'Other Opioid', 'Any Opioid']
Heroin
NaN    7182
Y      3472
Name: count, dtype: int64
-----
Heroin death certificate (DC)
NaN    9914
Y       740
Name: count, dtype: int64
-----
Cocaine
NaN    6799
Y      3855
Name: count, dtype: int64
-----
Fentanyl
Y           6923
NaN         3729
Y POPS         1
Y (PTCH)       1
Name: count, dtype: int64
-----
Fentanyl Analogue
NaN    9850
Y       804
Name: count, dtype: int64
-----
Oxycodone
NaN    9704
Y       950
Name: count, dtype: int64
-----
Oxymorphone
NaN    10497
Y        157
Name: count, dtype: int64
-----
Ethanol
NaN    7776
Y      2877
P         1
Name: count, dtype: int64
-----
Hydrocodone
NaN    10491
Y        163
Name:

In [5]:
heroine_analyse = deces_brut["Heroin"].fillna("vide")
heroine_certificat = deces_brut["Heroin death certificate (DC)"].fillna("vide")
pd.crosstab(heroine_analyse, heroine_certificat)

Heroin death certificate (DC),Y,vide
Heroin,,
Y,740,2732
vide,0,7182


**Constat 4 — Valeurs des colonnes de substances**

J'ai vérifié les 22 colonnes de `Heroin` à `Any Opioid` avec `value_counts(dropna=False)`.
15 colonnes ne contiennent que `Y` et du vide. 7 colonnes ont des valeurs bizarres :

- `Fentanyl` : `Y POPS` (1) et `Y (PTCH)` (1). Ça commence par Y, donc je compte comme détecté.
- `Gabapentin` : `y` (1). C'est un Y en minuscule, donc détecté.
- `Heroin/Morph/Codeine` : `y` (45). Pareil, détecté.
- `Any Opioid` : `N` (8). C'est un non, donc non détecté.
- `Ethanol` : `P` (1). Je ne sais pas ce que ça veut dire, donc je ne le compte pas comme détecté.
- `Morphine (Not Heroin)` : 3 notes écrites à la main (`STOLE MEDS`...). Ce ne sont pas des résultats d'analyse, donc non détecté.
- `Other Opioid` : 39 lignes avec un nom de substance au lieu de Y, dont certaines ne sont pas des opioïdes (PCP, kétamine). Je n'utilise pas cette colonne.

Je ne supprime aucune ligne : c'est seulement l'info sur la substance qui pose problème, pas le décès.
Attention pour le nettoyage : si je garde seulement les `Y` exacts, je perds 48 détections.

**Colonnes que je ne mets pas dans le top 5 (Q1) :**
- `Any Opioid`, `Heroin/Morph/Codeine` et `Opiate NOS` regroupent plusieurs substances. Ce ne sont pas des substances seules.
- `Other Opioid` et `Other` mélangent tout.
- `Heroin death certificate (DC)` : j'ai croisé cette colonne avec `Heroin`. Les 740 décès avec l'héroïne sur le certificat ont tous aussi une détection d'héroïne par l'analyse, et il y a 0 cas sur le certificat sans détection. Cette colonne n'ajoute rien, je garde `Heroin`.

Il y a aussi 2 732 décès où l'héroïne est détectée mais pas écrite sur le certificat. Ça veut potentiellement dire que l'héroïne n'a pas été retenu comme cause principale du décès. Donc dans ce projet, je compte des détections, pas des causes de décès. Je l’écrirai dans les limites.

In [6]:
deces_brut[["Date", "Date Type"]].head(10)

,Date,Date Type
0,05/29/2012,Date of death
1,06/27/2012,Date of death
2,03/24/2014,Date of death
3,12/31/2014,Date of death
4,01/16/2016,Date of death
5,06/13/2017,Date reported
6,10/20/2015,Date reported
7,02/02/2017,Date reported
8,07/03/2018,Date of death
9,05/08/2013,Date of death


In [7]:
deces_brut["Date Type"].value_counts()

Date Type
Date of death    8887
Date reported    1767
Name: count, dtype: int64

In [8]:
dates_test = pd.to_datetime(deces_brut["Date"], format="%m/%d/%Y", errors="coerce")
print(dates_test.isna().sum())
print(dates_test.min(), dates_test.max())

0
2012-01-01 00:00:00 2022-12-31 00:00:00


In [9]:
annee_test = dates_test.dt.year
pd.crosstab(annee_test, deces_brut["Date Type"], normalize="index").round(3)

Date Type,Date of death,Date reported
Date,,
2012,1.0,0.0
2013,1.0,0.0
2014,1.0,0.0
2015,0.0,1.0
2016,1.0,0.0
2017,0.0,1.0
2018,1.0,0.0
2019,1.0,0.0
2020,1.0,0.0


In [10]:
deces_brut["Sex"].value_counts(dropna=False)

Sex
Male      7890
Female    2755
NaN          9
Name: count, dtype: int64

In [11]:
deces_brut["Residence State"].value_counts(dropna=False)

Residence State
CT     8460
NaN    1898
NY       72
MA       60
FL       34
RI       18
NJ       17
NC       13
ME        8
PA        8
TX        6
CA        6
VA        5
NH        5
SC        4
CO        4
VT        4
AL        3
IL        3
OK        3
GA        3
LA        3
OH        2
WA        2
MD        2
TN        2
SD        1
IA        1
WV        1
DC        1
MI        1
MN        1
KY        1
IN        1
WY        1
Name: count, dtype: int64

In [12]:
deces_brut["Residence County"].value_counts(dropna=False)

Residence County
NEW HAVEN     2742
HARTFORD      2555
FAIRFIELD     1497
NaN           1168
NEW LONDON     865
              ... 
RENSSELAER       1
FREMONT          1
DUVAL            1
DELAWARE         1
SWEETWATER       1
Name: count, Length: 153, dtype: int64

In [13]:
comtes_ct = ["FAIRFIELD", "HARTFORD", "LITCHFIELD", "MIDDLESEX",
             "NEW HAVEN", "NEW LONDON", "TOLLAND", "WINDHAM"]
comte_normalise = deces_brut["Residence County"].str.strip().str.upper()
print(deces_brut["Residence County"].isin(comtes_ct).sum())
print(comte_normalise.isin(comtes_ct).sum())
comte_normalise[comte_normalise.isin(comtes_ct)].value_counts()

9175
9175


Residence County
NEW HAVEN     2742
HARTFORD      2555
FAIRFIELD     1497
NEW LONDON     865
LITCHFIELD     522
MIDDLESEX      396
WINDHAM        331
TOLLAND        267
Name: count, dtype: int64

In [14]:
dans_comte_ct = comte_normalise.isin(comtes_ct)
deces_brut.loc[dans_comte_ct, "Residence State"].value_counts(dropna=False)

Residence State
CT     8392
NaN     776
MA        4
VT        2
NJ        1
Name: count, dtype: int64

**Constat 5 — Les dates**

- La colonne `Date` est du texte au format mois/jour/année (ex. `05/29/2012`). Il n'y a pas d'heure.
- J'ai testé la conversion avec `to_datetime(format="%m/%d/%Y", errors="coerce")` : 0 date non convertie.
- Les dates vont du 1er janvier 2012 au 31 décembre 2022. Pas de date aberrante.
- Les lignes ne sont pas triées par date : il faudra trier avant de faire une courbe.

`Date Type` : 8 887 dates de décès et 1 767 dates de déclaration (16,6 %).
En croisant avec l'année, j'ai vu que ça dépend de l'année entière :
2015 et 2017 n'ont que des dates de déclaration, les autres années n'ont que des dates de décès.
Donc les 1 767 dates de déclaration correspondent à tous les décès de 2015 et 2017.

**Décision** : je garde toutes les dates. Si je gardais seulement les dates de décès,
2015 et 2017 disparaîtraient complètement et la courbe serait fausse.
- Par année : je suppose qu'une déclaration arrive au maximum environ 2 semaines après le décès (c'est une hypothèse).
  Seuls quelques décès de fin décembre pourraient changer d'année, donc c'est négligeable.
- Par mois : 2015 et 2017 sont moins précis. Je l'écrirai dans les limites.

**Constat 6 — Le sexe**

`Sex` : 7 890 hommes, 2 755 femmes, 9 vides. Pas de catégorie « inconnu » déjà présente.

**Décision** : je remplace les 9 vides par `Unknown` au lieu de supprimer les lignes.
Supprimer les lignes enlèverait ces décès de toutes les questions (substances, comtés, années).
Dans les graphiques par sexe, je n'afficherai pas la catégorie `Unknown` (9 décès sur 10 654).

**Constat 7 — Lieu de résidence**

`Residence State` : 8 460 résidents du Connecticut, 296 d'autres États (NY, MA, FL...), 1 898 vides.
`Residence County` : 153 valeurs différentes, 1 168 vides. Il y a des comtés d'autres États.
Pour la Q4, seuls les 8 comtés du Connecticut comptent. Les résidents d'autres États restent dans
le jeu de données pour les autres questions.
J'ai cherché les 8 comtés du Connecticut dans `Residence County` : 9 175 décès.
Le résultat est le même avant et après `strip()` + `upper()`, donc il n'y a pas de faute d'écriture.

En regardant l'État de ces 9 175 décès : 8 392 `CT`, 776 vides, et 7 dans un autre État
(4 MA, 2 VT, 1 NJ). Ces 7 sont des comtés avec le même nom dans un autre État.

**Décision pour la Q4** : je garde les décès dont le comté est un des 8 comtés du Connecticut
et dont l'État est `CT` ou vide, soit 9 168 décès (86 % du total).
Je garde les 776 sans État parce que leur comté porte le nom d’un comté du Connecticut. Quand l’État est connu, ce nom correspond à un autre État dans seulement 7 cas sur 8 399 (moins de 0,1 %). Parmi les 776, on peut donc s’attendre à moins d’une erreur.
Les autres décès ne sont pas supprimés, ils servent pour les autres questions.